# F1 Machine Learning — End-to-end Demo

This notebook walks through the full pipeline:

1. Load and clean the raw Kaggle F1 CSVs
2. Build per-(race, driver) features
3. Train three models: race winner, podium, and lap-pace regression
4. Inspect predictions for the most recent race
5. Show a few exploratory visualizations

Run `python scripts/clean_data.py`, `python scripts/build_features.py`, and
`python scripts/train_models.py` from the project root for the equivalent CLI flow.

In [ ]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

## 1. Load and clean the raw CSVs

In [ ]:
from src.data.data_cleaner import DataCleaner

cleaner = DataCleaner(PROJECT_ROOT / 'data' / 'raw')
tables = cleaner.run()
cleaner.get_summary()

## 2. Build the feature table

Every feature is computed using only races strictly *before* the target row,
so it is safe to use as model input without leakage.

In [ ]:
from src.features.feature_builder import build_feature_table

frames = build_feature_table(tables, min_year=2010)
df = frames.df
print(f'feature table shape: {df.shape}')
print(f'feature cols ({len(frames.feature_cols)}): {frames.feature_cols[:8]} ...')
df[['year', 'race_name', 'driver_name', 'grid', 'target_win',
    'driver_career_wins', 'driver_circuit_avg_position']].tail(10)

## 3. Train the three models

The most recent two seasons are held out as the test set.

In [ ]:
from src.models import (
    train_race_winner_model,
    train_podium_model,
    train_lap_pace_model,
)

winner = train_race_winner_model(df, frames.feature_cols, test_years=2)
podium = train_podium_model(df, frames.feature_cols, test_years=2)
pace = train_lap_pace_model(df, frames.feature_cols, test_years=2)

import pandas as pd
pd.DataFrame({'winner': winner.metrics, 'podium': podium.metrics, 'pace': pace.metrics}).round(4)

## 4. Predict the most recent race

In [ ]:
from src.models import (
    predict_race_winner_probs,
    predict_podium_probs,
    predict_lap_pace,
)

last_race_id = int(df.sort_values('date').iloc[-1]['raceId'])
race_df = df[df['raceId'] == last_race_id].copy()
race_name = race_df.iloc[0]['race_name']
year = int(race_df.iloc[0]['year'])
print(f'Predicting raceId={last_race_id} ({year} {race_name})')

winner_preds = predict_race_winner_probs(winner, race_df)
names = race_df[['driverId', 'driver_name', 'constructor_name', 'grid']].drop_duplicates('driverId')
winner_preds.merge(names, on='driverId').head(10)

In [ ]:
podium_preds = predict_podium_probs(podium, race_df).merge(names, on='driverId')
podium_preds.head(10)

In [ ]:
pace_preds = predict_lap_pace(pace, race_df).merge(names, on='driverId').sort_values('pace_rank')
pace_preds.head(10)

## 5. Exploratory visualizations

In [ ]:
from src.visualization import (
    plot_circuits_by_country,
    plot_constructor_wins_heatmap,
    plot_driver_podiums,
    plot_pit_stop_trends,
    plot_constructor_dominance,
)

plot_circuits_by_country(tables);
plot_constructor_wins_heatmap(tables);
plot_driver_podiums(tables);
plot_pit_stop_trends(tables);
plot_constructor_dominance(tables);

In [ ]:
from src.visualization import (
    world_map_of_circuits,
    driver_career_timeline,
    championship_standings_chart,
    feature_importance_plot,
    predicted_winner_panel,
)

world_map_of_circuits(tables['circuits']).show()
championship_standings_chart(tables, year=2023).show()
feature_importance_plot(winner.model, winner.feature_cols).show()
predicted_winner_panel(winner_preds, tables['drivers'], last_race_id).show()